# NLP Lab 2: Text Representation, N-Gram Language Modeling, and Shannon's Guessing Game

Welcome to Lab 2! In this session, we will shift our focus from cleaning text to representing and modeling it. We will explore how computers represent text numerically using Bag of Words and TF-IDF, how to build probabilistic N-gram language models from scratch, and finally, how to simulate Claude Shannon's famous next-word guessing game.

---

## 1. Setup and Preprocessing (Integrated from Lab 1)
First, let's import the necessary libraries and set up a full preprocessing pipeline using the tokenization, stop word removal, and lemmatization workflows built during Lab 1.

In [6]:
import re
import math
import nltk
from collections import defaultdict, Counter
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Ensure NLTK components are ready
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def full_preprocess(text, remove_stopwords=True, use_lemmatization=True):
    """
    Applies the complete preprocessing suite from Lab 1.
    Includes regex cleaning, tokenization, stop word stripping, and lemmatization.
    """
    # 1. Regex Cleaning (Lowercasing, HTML stripping, non-alpha removal)
    text = text.lower()
    text = re.sub(r'<[^>]+>', '', text) 
    text = re.sub(r'[^a-z\s]', '', text) 
    
    # 2. Tokenization
    tokens = word_tokenize(text)
    
    # 3. Stop Word Removal
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]
        
    # 4. Lemmatization
    if use_lemmatization:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
        
    return tokens

# Sample corpus for demonstration
corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "The lazy dog barks loudly at the moon.",
    "Foxes are quick and foxes are clever animals."
]

# Run the pipeline across the corpus
preprocessed_corpus = [full_preprocess(doc) for doc in corpus]
flat_tokens = [token for doc in preprocessed_corpus for token in doc]

print("Original Document 1 :", corpus[0])
print("Fully Preprocessed  :", preprocessed_corpus[0])

Original Document 1 : The quick brown fox jumps over the lazy dog.
Fully Preprocessed  : ['quick', 'brown', 'fox', 'jump', 'lazy', 'dog']


---

## 2. Text Representation: Bag of Words (BoW) & TF-IDF

> **A Quick Clarification:** While **Bag of Words (BoW)** and **TF-IDF** are text representation models used to convert documents into numerical vectors (often for classification or information retrieval), **N-grams** can be used within them to capture multi-word phrases. However, *N-gram Language Modeling* (which we will cover next) is a probabilistic sequence model designed to compute the likelihood of words occurring in order. Let's look at vector representations first.

### 2.1 Bag of Words (BoW)
The Bag of Words model discards grammar and word order, treating text purely as a "bag" of its word frequencies.

### 2.2 Term Frequency-Inverse Document Frequency (TF-IDF)
TF-IDF scales down terms that appear too frequently across all documents (like "the"), ensuring unique, highly informative words receive higher weights.

The mathematical formula for a term $t$ in a document $d$ within a corpus $D$ is:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

Where:
*   $\text{TF}(t, d) = \frac{\text{Count of } t \text{ in } d}{\text{Total words in } d}$
*   $\text{IDF}(t, D) = \log \left( \frac{|D|}{1 + |\{d \in D : t \in d\}|} \right)$

In [7]:
print("--- 1. Bag of Words Representation ---")
# Join preprocessed tokens back into space-separated strings for sklearn compatibility
raw_clean_docs = [" ".join(doc) for doc in preprocessed_corpus]

bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(raw_clean_docs)

print("Vocabulary Found :", bow_vectorizer.get_feature_names_out())
print("BoW Matrix Array :\n", bow_matrix.toarray())

print("\n--- 2. TF-IDF Representation ---")
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(raw_clean_docs)

print("TF-IDF Matrix Array:\n", np.round(tfidf_matrix.toarray(), 3))

--- 1. Bag of Words Representation ---
Vocabulary Found : ['animal' 'bark' 'brown' 'clever' 'dog' 'fox' 'jump' 'lazy' 'loudly'
 'moon' 'quick']
BoW Matrix Array :
 [[0 0 1 0 1 1 1 1 0 0 1]
 [0 1 0 0 1 0 0 1 1 1 0]
 [1 0 0 1 0 2 0 0 0 0 1]]

--- 2. TF-IDF Representation ---
TF-IDF Matrix Array:
 [[0.    0.    0.481 0.    0.366 0.366 0.481 0.366 0.    0.    0.366]
 [0.    0.49  0.    0.    0.373 0.    0.    0.373 0.49  0.49  0.   ]
 [0.452 0.    0.    0.452 0.    0.688 0.    0.    0.    0.    0.344]]


---

## 3. N-Gram Language Modeling

An **N-gram** is a contiguous sequence of $n$ items from a given sample of text. 
*   $n=1$: Unigram ("fox")
*   $n=2$: Bigram ("fox jump")
*   $n=3$: Trigram ("fox jump lazy")

Language models estimate the probability of a sequence of words. Using the **Markov Assumption**, we assume the probability of a word depends only on the preceding $n-1$ words.

The **Maximum Likelihood Estimation (MLE)** probability for a Bigram model is expressed as:

$$P_{MLE}(w_i | w_{i-1}) = \frac{C(w_{i-1} w_i)}{C(w_{i-1})}$$

Where $C(w_{i-1} w_i)$ is the count of the bigram sequence, and $C(w_{i-1})$ is the frequency of the history word.

In [8]:
def build_ngram_model(tokens, n=2):
    """Builds an N-gram probability lookup table from raw tokens."""
    model = defaultdict(Counter)
    
    # Track distributions across the historical contextual sliding frames
    for i in range(len(tokens) - n + 1):
        history = tuple(tokens[i:i+n-1])
        next_word = tokens[i+n-1]
        model[history][next_word] += 1
        
    # Convert absolute raw counts to MLE probabilities
    prob_model = defaultdict(dict)
    for history, context in model.items():
        total_count = sum(context.values())
        for next_word, count in context.items():
            prob_model[history][next_word] = count / total_count
            
    return prob_model

# Build a Bigram (n=2) model using an expanded dataset 
training_text = """
I love natural language processing because language processing is highly fun. 
I love machine learning models and I love coding execution frameworks inside Python. 
"""
# Keep stop words here so N-grams retain logical transitional sequences
tokens_train = full_preprocess(training_text, remove_stopwords=False, use_lemmatization=True)

bigram_model = build_ngram_model(tokens_train, n=2)

# Display sample bigram probabilities for the lemma history word 'love'
print("Bigram probabilities given historical lemma context ('love',):")
for word, prob in bigram_model[('love',)].items():
    print(f"  P({word} | love) = {prob:.3f}")

Bigram probabilities given historical lemma context ('love',):
  P(natural | love) = 0.333
  P(machine | love) = 0.333
  P(coding | love) = 0.333
